In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import xarray as xr
import harmonica as hm

In [2]:
#pip install harmonica

In [11]:
import rasterio
from rasterio.merge import merge
from glob import glob

files = glob("../../data/LiDAR/LiDAR_2026-06-05T15_32_32.740Z/*.tif")

src_files = [rasterio.open(f) for f in files]

mosaic, out_transform = merge(src_files)

out_meta = src_files[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform
})

merged_path = "merged_lidar.tif"

with rasterio.open(merged_path, "w", **out_meta) as dst:
    dst.write(mosaic)

for src in src_files:
    src.close()

print("Saved:", merged_path)

Saved: merged_lidar.tif


In [15]:
# =====================================================
# Setup gravity + merged LiDAR coordinates for correction
# =====================================================

import os
from glob import glob

import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge
from pyproj import Transformer

# -----------------------------
# 1. Paths
# -----------------------------
gravity_path = "../../data/gravity/Onyx_Gravity_Final_Bouguer_Corrected.csv"
lidar_folder = "../../data/LiDAR/LiDAR_2026-06-05T15_32_32.740Z"
merged_path = "merged_lidar.tif"

# -----------------------------
# 2. Load gravity data
# -----------------------------
df = pd.read_csv(gravity_path)
df = df.sort_values("station").reset_index(drop=True)

print("Gravity columns:")
print(df.columns.tolist())

# -----------------------------
# 3. Merge LiDAR tiles if merged file does not exist
# -----------------------------
if not os.path.exists(merged_path):
    files = glob(os.path.join(lidar_folder, "*.tif"))

    print("\nLiDAR files found:")
    for f in files:
        print(f)

    if len(files) == 0:
        raise FileNotFoundError("No LiDAR .tif files found. Check lidar_folder path.")

    src_files = [rasterio.open(f) for f in files]
    mosaic, out_transform = merge(src_files)

    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform
    })

    with rasterio.open(merged_path, "w", **out_meta) as dst:
        dst.write(mosaic)

    for src in src_files:
        src.close()

    print("\nSaved merged LiDAR:", merged_path)

else:
    print("\nMerged LiDAR already exists:", merged_path)

# -----------------------------
# 4. Check merged LiDAR CRS/bounds
# -----------------------------
with rasterio.open(merged_path) as src:
    lidar_crs = src.crs
    lidar_bounds = src.bounds
    elev = src.read(1)

print("\nLiDAR CRS:")
print(lidar_crs)

print("\nLiDAR bounds:")
print(lidar_bounds)

print("\nLiDAR elevation range, feet:")
print(np.nanmin(elev), np.nanmax(elev))

# -----------------------------
# 5. Project gravity lat/lon into LiDAR CRS
# -----------------------------
transformer = Transformer.from_crs("EPSG:4326", lidar_crs, always_xy=True)

df["lidar_x_ft"], df["lidar_y_ft"] = transformer.transform(
    df["longitude"].values,
    df["latitude"].values
)

# US survey foot to meters
FT_TO_M = 1200 / 3937

df["lidar_x_m"] = df["lidar_x_ft"] * FT_TO_M
df["lidar_y_m"] = df["lidar_y_ft"] * FT_TO_M

# Use these in the terrain correction code
lidar_path = merged_path
x_col = "lidar_x_m"
y_col = "lidar_y_m"

print("\nGravity station LiDAR-coordinate range, feet:")
print("x:", df["lidar_x_ft"].min(), "to", df["lidar_x_ft"].max())
print("y:", df["lidar_y_ft"].min(), "to", df["lidar_y_ft"].max())

print("\nUse these for terrain correction:")
print("lidar_path =", lidar_path)
print("x_col =", x_col)
print("y_col =", y_col)

df.head()

Gravity columns:
['station', 'gravity_final_mgal', 'elevation_m', 'latitude', 'longitude', 'free_air_correction_mgal', 'bouguer_correction_mgal', 'gravity_tied_mgal', 'instrument']

Merged LiDAR already exists: merged_lidar.tif

LiDAR CRS:
PROJCS["NAD83(2011) / Colorado North (ftUS)",GEOGCS["NAD83(2011)",DATUM["unnamed",SPHEROID["unnamed",6378137,298.257222101004],AUTHORITY["EPSG","1116"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",39.3333333333333],PARAMETER["central_meridian",-105.5],PARAMETER["standard_parallel_1",39.7166666666667],PARAMETER["standard_parallel_2",40.7833333333333],PARAMETER["false_easting",3000000.00031608],PARAMETER["false_northing",999999.999996],UNIT["US survey foot",0.304800609601219,AUTHORITY["EPSG","9003"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

LiDAR bounds:
BoundingBox(left=2623000.0, bottom=1421000.0, right=2629000.0, top=1427000.0)

LiDAR 

ERROR 1: PROJ: internal_proj_identify: /opt/homebrew/Caskroom/miniconda/base/envs/gpgn-318/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 5 whereas a number >= 6 is expected. It comes from another PROJ installation.


,station,gravity_final_mgal,elevation_m,latitude,longitude,free_air_correction_mgal,bouguer_correction_mgal,gravity_tied_mgal,instrument,lidar_x_ft,lidar_y_ft,lidar_x_m,lidar_y_m
0,1.0,0.589314,2112.468,40.478666,-106.835495,0.803286,0.291414,0.077442,CG5-2,2.628496e+06,1.420016e+06,801167.329383,432821.729285
1,2.0,0.423992,2112.006,40.478675,-106.835498,0.660713,0.239692,0.002971,CG5-2,2.628496e+06,1.420019e+06,801167.094979,432822.775598
2,3.0,0.423230,2111.569,40.478684,-106.835500,0.525854,0.190768,0.088144,CG5-2,2.628495e+06,1.420023e+06,801166.899932,432823.732477
3,4.0,0.458692,2111.692,40.478692,-106.835503,0.563812,0.204538,0.099418,CG5-2,2.628494e+06,1.420026e+06,801166.690088,432824.664037
4,5.0,0.507517,2111.862,40.478702,-106.835504,0.616274,0.223570,0.114813,CG5-2,2.628494e+06,1.420029e+06,801166.597504,432825.723762
